# 05_reviews_llm — TV5 (Phan Ngọc Huy)
Notebook khám phá & kiểm chứng độc lập cho phần Reviews, Keyword Baseline, Gemini LLM, Evaluation và KPI 7 (Guest Experience Issue Rate). Logic chính đặt tại `src/airbnb/`.

In [1]:
import importlib
import json
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import airbnb.reviews as reviews_mod
import airbnb.baseline as baseline_mod
import airbnb.llm as llm_mod
import airbnb.evaluate as evaluate_mod
import airbnb.review_kpi as review_kpi_mod
import airbnb.viz_reviews as viz_reviews_mod

for mod in (reviews_mod, baseline_mod, llm_mod, evaluate_mod, review_kpi_mod, viz_reviews_mod):
    importlib.reload(mod)

from airbnb.io import load_config
from airbnb.reviews import collect, clean_comment, detect_language_simple, extract_aspect_keywords
from airbnb.baseline import predict, predict_with_aspects
from airbnb.llm import validate, estimate_cost, SENTIMENTS
from airbnb.evaluate import evaluate, evaluate_detailed
from airbnb.review_kpi import guest_kpi

cfg, _ = load_config(ROOT / "configs/toronto.yaml")
print("Loaded config for:", cfg["city"], "| LLM model:", cfg["llm"]["model"])

Loaded config for: toronto | LLM model: gemini-3.5-flash-lite


In [2]:
# Kiểm tra gói 20 nhãn tay của TV5 và đánh giá Baseline
tv5_packet = json.loads((ROOT / "data/samples/labeling/tv5.json").read_text(encoding="utf-8"))
tv5_labels = tv5_packet["labels"]
preds_detailed = [predict_with_aspects(r, cfg["llm"]["aspects"]) for r in tv5_labels]
eval_detail = evaluate_detailed(preds_detailed, tv5_labels, "baseline")
print("Evaluation summary (TV5 20 labeled reviews):", eval_detail["summary"])
print("Per-class metrics:", json.dumps(eval_detail["per_class"], indent=2))

Evaluation summary (TV5 20 labeled reviews): {'source': 'baseline', 'paired_n': 20, 'manual_n': 20, 'accuracy': 1.0, 'macro_f1': 1.0, 'meets_100_paired': False}
Per-class metrics: {
  "positive": {
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "support": 17
  },
  "mixed": {
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "support": 2
  },
  "negative": {
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "support": 1
  }
}


In [3]:
# Tính KPI 7 (Guest Experience Issue Rate) và ước tính chi phí Gemini API
kpis = guest_kpi(preds_detailed, [], cfg)
for row in kpis[:6]:
    print(f"{row['metric']}: {row['value']:.4f} (n={row['sample_n']}, scope={row['scope']})")

cost = estimate_cost(tv5_labels, cfg)
print("\nCost estimate:", json.dumps(cost, indent=2))

guest_issue_rate: 0.1500 (n=20, scope=assigned_llm_sample)
positive_sentiment_rate: 0.8500 (n=20, scope=assigned_llm_sample)
mixed_sentiment_rate: 0.1000 (n=20, scope=assigned_llm_sample)
negative_sentiment_rate: 0.0500 (n=20, scope=assigned_llm_sample)
negative_aspect_location_rate: 0.0000 (n=20, scope=assigned_llm_sample)
positive_aspect_location_rate: 0.4500 (n=20, scope=assigned_llm_sample)

Cost estimate: {
  "model": "gemini-3.5-flash-lite",
  "avg_review_chars": 295.6,
  "avg_input_tokens_per_review": 399.7,
  "avg_output_tokens_per_review": 45.0,
  "input_price_per_1m_usd": 0.15,
  "output_price_per_1m_usd": 0.6,
  "sample_estimate": {
    "review_count": 1000,
    "estimated_input_tokens": 399662,
    "estimated_output_tokens": 45000,
    "estimated_cost_usd": 0.086949
  },
  "full_assigned_estimate": {
    "review_count": 5000,
    "estimated_input_tokens": 1998312,
    "estimated_output_tokens": 225000,
    "estimated_cost_usd": 0.434747
  }
}
